In [ ]:
# INQUIRE toxicity-driven prioritization - Part 3: Visualization
"""
Reads only the files saved by parts 1-2. Set the paths in the first cell, then run all cells.
Before running: in part 2, run the cross-endpoint statistics summary and compound summary cells.
Set IDENTIFIED_ONLY in the setup cell: False = all features (incl. unknowns), True = identified compounds only
(figures saved with the suffix "_identified"). Run the notebook once per setting to get both sets.
"""


In [ ]:
# INQUIRE toxicity-driven prioritization - Part 3: Visualization
# Standalone: reads only files saved by parts 1 (preprocessing) and 2 (analysis); no variables from parts 1-2 are needed
# - Fig 1. Model performance per endpoint (cross-validated Q² / AUC)
# - Fig 2. Measured vs. cross-validated predicted (regression) and ROC curves (classification)
# - Fig 3. Chemical class enrichment among the prioritized compounds
# - Fig 4. Endpoint vs. concentration for the top compounds of each endpoint
# - Fig 5. Cross-endpoint heatmap of the top compounds (global ranking)
# Output: OUT_DIR / "figures": .pdf (vector, editable text), .svg, .png (600 dpi) + figure_notes.txt (caption drafts)

# Setup
import warnings
import logging
from datetime import datetime
from pathlib import Path
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from scipy.stats import spearmanr, mannwhitneyu, hypergeom
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import r2_score, roc_auc_score, roc_curve
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# log_file = f"inquire_tox_ind_prio_figures_{datetime.now().strftime('%y%m%d')}.log"
log_file = f"inquire_tip_pt3_visualization_{datetime.now().strftime('%y%m%d_%H%M%S')}.log" # date + time: a new run never overwrites the log of an earlier run
logging.basicConfig(level=logging.INFO, format="%(message)s",
                    handlers=[logging.FileHandler(log_file, mode="w"), logging.StreamHandler()], force=True)

logging.getLogger("fontTools").setLevel(logging.WARNING) # font subsetting messages when saving pdf
logging.getLogger("matplotlib").setLevel(logging.WARNING)

def majorsep():
    logging.info("=" * 100)

def minorsep():
    logging.info("-" * 100)

# Paths
DATA_DIR = Path("../data") # working folder (data/ next to notebooks/); or the full path, e.g. Path(r"D:/projects/INQUIRE_TIP/data")
OUT_DIR = DATA_DIR / "inquire_tip_outputs"
PT1_DATE = None # yymmdd prefix of the part 1 run used in part 2 (same as PT1_DATE there); None = most recent part 1 run
TOX_DUST_FILE = DATA_DIR / "tox_vs_dust_merged.csv" # raw merged tox vs. dust data (for concentrations); same file as in part 1
META_CLASS_FILE = DATA_DIR / "feature_metadata_classified.xlsx" # classified feature metadata; same file as in part 1
SAMPLE_COL_RAW = "sample_base" # sample ID column in the raw file

MODELS_DIR = OUT_DIR / "endpoint_models"
SUMMARY_DIR = OUT_DIR / "cross_endpoint_summary"
FIG_DIR = OUT_DIR / "inquire_tip_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Settings (must match parts 1-2)
NEG_HANDLING = "abs" # below-LOD values (negative numbers): "abs" = |value|, "zero" = not detected
GLOBAL_SEED = 1
N_SPLITS = 5 # CV folds (Fig 2)
RECOMPUTE_CV = False # False = reuse saved cross-validated predictions (Fig 2) if present

# Figure settings
N_TOP_ENRICH = 100 # Fig 3: top compounds per endpoint tested for class enrichment
MIN_CLASS_SIZE = 15 # Fig 3: classes with fewer features (on average) are pooled as "Other classes"
N_TOP_COMPOUNDS = 6 # Fig 4: compounds per endpoint
N_TOP_HEATMAP = 30 # Fig 5: compounds from the global ranking
FIG_WIDTH_FULL = 7.09 # in (180 mm, double column)
FIG_FORMATS = ["pdf", "svg", "png"]

# Unknown features
IDENTIFIED_ONLY = False # True = Figs 3-5 show identified compounds only; files get the suffix "_identified"
UNKNOWN_NAME_TEXT = "Feature" # feature_name containing this text = unknown/unidentified feature (case-sensitive)
NO_NAME_IS_UNKNOWN = True # features without a feature_name (or not in the metadata) also count as unknown
FIG_SUFFIX = "_identified" if IDENTIFIED_ONLY else ""

# Endpoints
ENDPOINTS = {
    "cell_viability": "reduced cell viability/mg BEAS-2B (effect cut-off = 20%)",
    "IL6_0.1": "relative IL-6 expression (BEAS-2B 0.1 mg/ml)",
    "IL6_1.0": "relative IL-6 expression (BEAS-2B 1 mg/ml)",
    "IL8_0.1": "relative IL-8 expression (BEAS-2B 0.1 mg/ml)",
    "IL8_1.0": "relative IL-8 expression (BEAS-2B 1 mg/ml)",
    "andro": "andro (ng/g)",
    "antiandro": "antiandro (µg/g)",
    "estro": "estro (pg/g)",
    "antiestro": "antiestro (ng/g)",
    "AhR": "AhR (ng/g)",
    "TR": "TR agonism (ng/g)",
}
ENDPOINT_LABELS = { # short labels for figures
    "cell_viability": "Cytotoxicity", "IL6_0.1": "IL-6 (0.1 mg/ml)", "IL6_1.0": "IL-6 (1 mg/ml)",
    "IL8_0.1": "IL-8 (0.1 mg/ml)", "IL8_1.0": "IL-8 (1 mg/ml)", "andro": "Androgenic", "antiandro": "Anti-androgenic",
    "estro": "Estrogenic", "antiestro": "Anti-estrogenic", "AhR": "AhR", "TR": "TR agonism",
}
ENDPOINT_UNITS = {
    "cell_viability": "% reduced viability per mg", "IL6_0.1": "% of control", "IL6_1.0": "% of control",
    "IL8_0.1": "% of control", "IL8_1.0": "% of control", "andro": "ng/g", "antiandro": "µg/g",
    "estro": "pg/g", "antiestro": "ng/g", "AhR": "ng/g", "TR": "ng/g",
}
ENDPOINT_BLOCKS = { # same as in part 2
    "cytotoxicity": ["cell_viability"],
    "inflammation": ["IL6_0.1", "IL6_1.0", "IL8_0.1", "IL8_1.0"],
    "AhR": ["AhR"],
    "androgen_receptor": ["andro", "antiandro"],
    "estrogen_receptor": ["estro", "antiestro"],
    "thyroid_receptor": ["TR"],
}
endpoint_order = list(ENDPOINTS)
endpoint_to_block = {e: b for b, eps in ENDPOINT_BLOCKS.items() for e in eps}

# Colors: categorical palette in fixed order (colorblind-checked), one color per endpoint block in all figures
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLOCK_COLORS = {b: SERIES[i] for i, b in enumerate(ENDPOINT_BLOCKS)}
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#8a8985" # text: primary, secondary, muted
GRID, NEUTRAL, EXCLUDED = "#e6e5e1", "#f0efec", "#b9b7b0"
BLUE_RAMP = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
             "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ_CMAP = LinearSegmentedColormap.from_list("seq_blue", BLUE_RAMP)
ENRICH_CMAP = LinearSegmentedColormap.from_list( # neutral gray (no enrichment) -> red (strong enrichment)
    "seq_red", [NEUTRAL, "#f6c7c5", "#f2aaa8", "#ea7d7b", "#e34948", "#c13434", "#a32a2a"])

def endpoint_color(ep):
    return BLOCK_COLORS.get(endpoint_to_block.get(ep, ""), SERIES[6])

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5,
    "legend.fontsize": 6.5, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "grid.color": GRID, "grid.linewidth": 0.5, "legend.frameon": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none", # editable text in pdf/svg
    "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.pad_inches": 0.03, "figure.dpi": 120,
})

# def save_fig(fig, name):
#     for fmt in FIG_FORMATS:
#         fig.savefig(FIG_DIR / f"{name}.{fmt}")
#     logging.info(f" - Saved {name} ({', '.join(FIG_FORMATS)})")
def save_fig(fig, name, shared=False):
    """shared=True: figure does not depend on IDENTIFIED_ONLY (Figs 1-2), saved without suffix"""
    name = name if shared else f"{name}{FIG_SUFFIX}"
    for fmt in FIG_FORMATS:
        fig.savefig(FIG_DIR / f"{name}.{fmt}")
    logging.info(f" - Saved {name} ({', '.join(FIG_FORMATS)})")

FIG_NOTES = {}
def save_notes(key, text):
    """Caption draft / key numbers per figure, collected in figure_notes.txt"""
    FIG_NOTES[key] = text.strip()
    # with open(FIG_DIR / "figure_notes.txt", "w", encoding="utf-8") as fh:
    with open(FIG_DIR / f"figure_notes{FIG_SUFFIX}.txt", "w", encoding="utf-8") as fh:
        for k in sorted(FIG_NOTES):
            fh.write(f"{k}\n{'-' * len(k)}\n{FIG_NOTES[k]}\n\n")

def fmt_p(p, n_perm=None):
    if pd.isna(p):
        return "n/a"
    if n_perm and p <= 1 / (n_perm + 1) + 1e-9:
        return f"≤ {1 / (n_perm + 1):.2g}"
    return f"= {p:.2g}" if p < 0.1 else f"= {p:.2f}"

majorsep()
logging.info(f"Figure setup done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logging.info(f"\nFigures folder: \n{FIG_DIR}")
logging.info(f" - Identified compounds only: {IDENTIFIED_ONLY}" + (f" (files saved with suffix '{FIG_SUFFIX}')" if IDENTIFIED_ONLY else ""))
majorsep()


In [ ]:
# Load saved outputs
majorsep()
logging.info(f"Loading saved outputs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

def latest_file(folder, pattern):
    files = sorted(Path(folder).glob(pattern))
    if not files:
        raise FileNotFoundError(f"No file matching '{pattern}' in {folder}")
    return files[-1]

# Endpoint model summary (part 2, new statistics summary cell)
stats_file = SUMMARY_DIR / "endpoint_model_summary.csv"
if not stats_file.exists():
    raise FileNotFoundError(f"{stats_file} not found: run the cross-endpoint statistics summary cell (part 2) first")
stats = pd.read_csv(stats_file)
stats["Endpoint"] = stats["Endpoint"].astype(str)
stats = stats.set_index("Endpoint")
needed = ["Task", "Target", "Metric", "Elastic Net CV metric mean", "Elastic Net CV metric SD", "Elastic Net best l1_ratio",
          "Included in compound ranking", "Minimum CV metric required"]
missing = [c for c in needed if c not in stats.columns]
if missing:
    # raise ValueError(f"Columns missing in {stats_file.name}: {missing}. Rerun the NEW (261017) statistics summary cell.")
    raise ValueError(f"Columns missing in {stats_file.name}: {missing}. Rerun the cross-endpoint statistics summary cell (part 2).")

endpoints_all = [e for e in endpoint_order if e in stats.index] + [e for e in stats.index if e not in endpoint_order]
INCLUDED = [e for e in endpoints_all if stats.loc[e, "Included in compound ranking"] == "Yes"]
logging.info(f" - Endpoints in summary: {endpoints_all}")
logging.info(f" - Endpoints used in the compound ranking: {INCLUDED}")

# Compound rankings
global_file = SUMMARY_DIR / "compound_global_summary_w_metadata.csv"
global_summary = pd.read_csv(global_file, low_memory=False) if global_file.exists() else None
if global_summary is None:
    logging.warning(f" - {global_file.name} not found: Fig 5 will be skipped")

fsaci = {}
for e in endpoints_all:
    f = MODELS_DIR / e / "fsaci_rank.csv"
    if f.exists():
        tmp = pd.read_csv(f)
        tmp["ML_feature_id"] = tmp["ML_feature_id"].astype(str).str.strip()
        fsaci[e] = tmp
    else:
        logging.warning(f" - {f} not found")
logging.info(f" - FSACI rankings loaded: {list(fsaci)}")

# Feature metadata
meta = pd.read_excel(META_CLASS_FILE, engine="openpyxl")
meta.columns = [str(c).strip() for c in meta.columns]
meta["ML_feature_id"] = meta["ML_feature_id"].astype(str).str.strip()
meta = meta.drop_duplicates("ML_feature_id").set_index("ML_feature_id")
if "class" not in meta.columns:
    meta["class"] = np.nan
meta["class"] = meta["class"].fillna("Unclassified").astype(str).str.strip().replace("", "Unclassified")
logging.info(f" - Metadata: {len(meta):,} features, {meta['class'].nunique()} classes")

# Identified vs. unknown features
names_meta = meta["feature_name"] if "feature_name" in meta.columns else pd.Series(np.nan, index=meta.index)
no_name = names_meta.isna() | names_meta.astype(str).str.strip().eq("")
unknown_by_name = names_meta.astype(str).str.contains(UNKNOWN_NAME_TEXT, case=True, regex=False) & ~no_name
UNKNOWN_IDS = set(meta.index[unknown_by_name | (no_name if NO_NAME_IS_UNKNOWN else False)])
IDENTIFIED_IDS = set(meta.index) - UNKNOWN_IDS
logging.info(f" - Metadata features: {len(IDENTIFIED_IDS):,} identified, {int(unknown_by_name.sum()):,} unknown "
             f"('{UNKNOWN_NAME_TEXT}' in feature_name), {int(no_name.sum()):,} without feature_name"
             + (" (counted as unknown)" if NO_NAME_IS_UNKNOWN else ""))

def is_identified(fid):
    """True if the feature is an identified compound (features not in the metadata count as unknown)."""
    return fid in IDENTIFIED_IDS or (not NO_NAME_IS_UNKNOWN and fid not in UNKNOWN_IDS)

def keep_mask(ids):
    """Boolean mask of features to show: all, or identified only (IDENTIFIED_ONLY = True)."""
    ids = list(ids)
    return np.array([is_identified(f) for f in ids], dtype=bool) if IDENTIFIED_ONLY else np.ones(len(ids), dtype=bool)

def feat_name(fid, width=None):
    name = meta["feature_name"].get(fid, np.nan) if "feature_name" in meta.columns else np.nan
    name = fid if pd.isna(name) or str(name).strip() == "" else str(name).strip()
    return textwrap.shorten(name, width=width, placeholder="…") if width else name

# Tox endpoints (part 1 output)
# y_file = latest_file(OUT_DIR, "*_tox_endpoints_unscaled.csv")
y_file = OUT_DIR / f"{PT1_DATE}_tox_endpoints_unscaled.csv" if PT1_DATE else latest_file(OUT_DIR, "*_tox_endpoints_unscaled.csv")
if not y_file.exists():
    raise FileNotFoundError(f"{y_file.name} not found in {OUT_DIR}: check PT1_DATE")
Y = pd.read_csv(y_file, index_col=0)
Y.index = Y.index.astype(str).str.strip()
Y = Y.apply(pd.to_numeric, errors="coerce")
logging.info(f" - Tox endpoints: {y_file.name} | {Y.shape}")

# Raw dust concentrations (only features that were modeled)
model_feats = sorted(set().union(*[set(d["ML_feature_id"]) for d in fsaci.values()])) if fsaci else []
raw = pd.read_csv(TOX_DUST_FILE, low_memory=False)
raw.columns = [str(c).strip() for c in raw.columns]
if SAMPLE_COL_RAW not in raw.columns:
    raise ValueError(f"Sample column '{SAMPLE_COL_RAW}' not found in {TOX_DUST_FILE.name}")
raw[SAMPLE_COL_RAW] = raw[SAMPLE_COL_RAW].astype(str).str.strip()
raw = raw[raw[SAMPLE_COL_RAW].ne("nan")].drop_duplicates(SAMPLE_COL_RAW).set_index(SAMPLE_COL_RAW)
missing_feats = [f for f in model_feats if f not in raw.columns]
if missing_feats:
    logging.warning(f" - {len(missing_feats)} modeled features not found in the raw file, e.g. {missing_feats[:5]}")
dust_raw = raw[[f for f in model_feats if f in raw.columns]].apply(pd.to_numeric, errors="coerce")

# Identified vs. unknown among the modeled and prioritized features
model_ident = sum(is_identified(f) for f in model_feats)
logging.info(f" - Modeled features: {len(model_feats):,} | identified: {model_ident:,} | unknown: {len(model_feats) - model_ident:,}")
for e, d_ in fsaci.items():
    pos_ = d_.loc[d_["FSACI"] > 0, "ML_feature_id"]
    logging.info(f"   {e}: prioritized (FSACI > 0) {len(pos_):,}, of which identified {sum(is_identified(f) for f in pos_):,}")

samples = Y.index.intersection(dust_raw.index)
Y, dust_raw = Y.loc[samples], dust_raw.loc[samples]
logging.info(f" - Dust data: {dust_raw.shape[1]:,} modeled features | samples in both: {len(samples)}")

minorsep()
logging.info(f"Done loading saved outputs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Rebuild the modeling data (same steps as part 1)
#  - NaN -> 0; detected = value > 0; below-LOD (negative) -> |value| (NEG_HANDLING = "abs") or 0
#  - log10(x + c), c = half the smallest detected value per feature (all samples)
#  - per endpoint: samples with an endpoint value, z-scaled as R scale(), features from that endpoint's model
majorsep()
logging.info(f"Rebuilding modeling data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

DUST0 = dust_raw.fillna(0)
DET = DUST0 > 0 # detected
BELOW_LOD = DUST0 < 0 # below-LOD estimates (GCxGC-MS)
DUST_CONC = DUST0.abs() if NEG_HANDLING == "abs" else DUST0.clip(lower=0)
LOG_OFFSET = DUST_CONC.where(DET).min() / 2
DUST_LOG = np.log10(DUST_CONC + LOG_OFFSET)

def endpoint_data(ep):
    """Returns dict with X (scaled), y_raw, y_model, task, target label - as used in part 2."""
    task = str(stats.loc[ep, "Task"])
    target = str(stats.loc[ep, "Target"])
    y_raw = Y[ENDPOINTS[ep]].dropna().astype(float)
    feats = [f for f in fsaci[ep]["ML_feature_id"] if f in DUST_LOG.columns]
    Xl = DUST_LOG.loc[y_raw.index, feats]
    sd = Xl.std(ddof=1)
    keep = sd.index[(sd > 0) & sd.notna()]
    X = (Xl[keep] - Xl[keep].mean()) / sd[keep]
    lab, unit = ENDPOINT_LABELS.get(ep, ep), ENDPOINT_UNITS.get(ep, "")
    if task == "classification":
        if "median" in target:
            cut = float(y_raw.median())
            y_model = (y_raw > cut).astype(float).values
        else:
            cut = float(y_raw.min())
            y_model = (~np.isclose(y_raw.values, cut)).astype(float)
        y_label = f"{lab} (active = above {cut:.3g} {unit})"
    elif target.startswith("log10"):
        off = float(y_raw[y_raw > 0].min() / 2) if (y_raw <= 0).any() else 0.0
        y_model = np.log10(y_raw.values + off)
        y_label = f"log10({lab} + {off:.3g}) [{unit}]" if off > 0 else f"log10 {lab} [{unit}]"
    else:
        y_model = y_raw.values
        y_label = f"{lab} [{unit}]"
    return dict(X=X, y_raw=y_raw, y_model=y_model, task=task, y_label=y_label)

# QC: sample and feature numbers vs. part 2 (modeling)
qc = []
for ep in [e for e in endpoints_all if e in fsaci]:
    d = endpoint_data(ep)
    qc.append([ep, len(d["y_raw"]), stats.loc[ep].get("Endpoint count", np.nan), d["X"].shape[1],
               stats.loc[ep].get("Total features", np.nan)])
# qc = pd.DataFrame(qc, columns=["endpoint", "samples", "samples (part 1)", "features", "features (part 1)"])
qc = pd.DataFrame(qc, columns=["endpoint", "samples", "samples (part 2)", "features", "features (part 2)"])
display(qc)
# mismatch = qc[(qc["samples"] != qc["samples (part 1)"]) | (qc["features"] != qc["features (part 1)"])]
mismatch = qc[(qc["samples"] != qc["samples (part 2)"]) | (qc["features"] != qc["features (part 2)"])]
if len(mismatch):
    # logging.warning(f" - Different from part 1 for: {mismatch['endpoint'].tolist()} (check NEG_HANDLING / input files)")
    logging.warning(f" - Different from part 2 for: {mismatch['endpoint'].tolist()} (check NEG_HANDLING / PT1_DATE / input files)")
else:
    # logging.info(f" - Samples and features match part 1 for all endpoints")
    logging.info(f" - Samples and features match part 2 for all endpoints")

minorsep()
logging.info(f"Done rebuilding modeling data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Fig 1: Model performance per endpoint
# Does not depend on IDENTIFIED_ONLY (models use all features): always saved without suffix
majorsep()
logging.info(f"Fig 1: model performance | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

panels = []
for task, metric, chance in [("regression", "Q²", 0.0), ("classification", "AUC", 0.5)]:
    eps = [e for e in endpoints_all if stats.loc[e, "Task"] == task]
    if eps:
        panels.append((task, metric, chance, eps))

n_rows_total = sum(len(p[3]) for p in panels)
fig, axes = plt.subplots(len(panels), 1, figsize=(FIG_WIDTH_FULL * 0.72, 0.30 * n_rows_total + 0.55 * len(panels) + 0.5),
                         gridspec_kw={"height_ratios": [len(p[3]) for p in panels]}, squeeze=False)
axes = axes[:, 0]

for ax, (task, metric, chance, eps), letter in zip(axes, panels, "abcdef"):
    n_perm = int(stats["N permutations"].dropna().iloc[0]) if "N permutations" in stats.columns and stats["N permutations"].notna().any() else None
    ypos = np.arange(len(eps))[::-1]
    vals = []
    for y, e in zip(ypos, eps):
        r = stats.loc[e]
        inc = r["Included in compound ranking"] == "Yes"
        col = endpoint_color(e) if inc else EXCLUDED
        for mean_col, sd_col, marker, dy, filled in [("Elastic Net CV metric mean", "Elastic Net CV metric SD", "o", 0.13, True),
                                                      ("XGBoost CV metric mean", "XGBoost CV metric SD", "D", -0.13, False)]:
            m, s = r.get(mean_col, np.nan), r.get(sd_col, np.nan)
            if pd.isna(m):
                continue
            vals.append(m)
            ax.errorbar(m, y + dy, xerr=0 if pd.isna(s) else s, fmt=marker, ms=5 if filled else 4.2,
                        mfc=col if filled else "white", mec=col if not filled else "white", mew=1.2 if not filled else 0.6,
                        ecolor=col, elinewidth=1.0, capsize=0, zorder=3)
        p_txt = f"Elastic Net p {fmt_p(r.get('Elastic Net permutation p-value'), n_perm)}   XGBoost p {fmt_p(r.get('XGBoost permutation p-value'), n_perm)}"
        ax.text(1.02, y, p_txt, transform=ax.get_yaxis_transform(), va="center", ha="left", fontsize=6,
                color=INK2 if inc else MUTED)
    min_req = stats.loc[eps, "Minimum CV metric required"].dropna()
    ax.axvline(chance, color=INK2, lw=0.8, zorder=1)
    if len(min_req):
        ax.axvline(float(min_req.iloc[0]), color=INK2, lw=0.8, ls=(0, (3, 2)), zorder=1)
    ax.set_yticks(ypos)
    ax.set_yticklabels([ENDPOINT_LABELS.get(e, e) for e in eps])
    for tl, e in zip(ax.get_yticklabels(), eps):
        if stats.loc[e, "Included in compound ranking"] != "Yes":
            tl.set_color(MUTED)
    lo = min(vals + [chance]) if vals else chance
    hi = max(vals + [float(min_req.iloc[0]) if len(min_req) else chance]) if vals else chance + 0.5
    span = hi - lo if hi > lo else 0.5
    ax.set_xlim(lo - 0.12 * span, hi + 0.12 * span)
    ax.set_ylim(-0.6, len(eps) - 0.4)
    ax.grid(axis="x", zorder=0)
    ax.set_axisbelow(True)
    ax.set_xlabel(f"Cross-validated {metric} ({'regression' if task == 'regression' else 'classification: active vs. inactive'})")
    ax.text(-0.02, 1.02, letter, transform=ax.transAxes, fontweight="bold", fontsize=9, ha="right", va="bottom")

handles = [Line2D([], [], marker="o", ls="", ms=5, mfc=INK2, mec="white", label="Elastic Net (mean ± SD of CV repeats)"),
           Line2D([], [], marker="D", ls="", ms=4.2, mfc="white", mec=INK2, mew=1.2, label="XGBoost"),
           Line2D([], [], color=INK2, lw=0.8, label="chance level"),
           Line2D([], [], color=INK2, lw=0.8, ls=(0, (3, 2)), label="minimum for compound ranking")]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
fig.tight_layout(h_pad=1.2)
# save_fig(fig, "fig1_model_performance")
save_fig(fig, "fig1_model_performance", shared=True) # same with or without unknown features
plt.show()

inc_txt = ", ".join(f"{ENDPOINT_LABELS.get(e, e)} ({stats.loc[e, 'Metric']} = {stats.loc[e, 'Best CV metric (models with signal)']:.2f})"
                    for e in INCLUDED if "Best CV metric (models with signal)" in stats.columns)
save_notes("Fig 1", f"""
Cross-validated performance of the Elastic Net (circles) and XGBoost (open diamonds) models per tox endpoint
(mean ± SD over the CV repeats; {N_SPLITS}-fold CV). (a) Regression endpoints, Q²; (b) classification endpoints
(active vs. inactive), ROC AUC. Solid line: chance level; dashed line: minimum effect size for the compound ranking.
Colored = endpoint used for the compound ranking; gray = not used. Permutation p-values on the right.
Used: {inc_txt}.
""")
logging.info(f"Done Fig 1 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Fig 2: Measured vs. cross-validated predicted (regression) / ROC curves (classification)
# Cross-validated predictions are recomputed for the Elastic Net with the settings of part 2 (alpha re-tuned
# inside every training fold, l1_ratio from part 2) for one CV repeat (= repeat 1 of part 2) and cached.
# Does not depend on IDENTIFIED_ONLY (models use all features): always saved without suffix
majorsep()
logging.info(f"Fig 2: measured vs. predicted / ROC | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

CV_DIR = FIG_DIR / "cv_predictions"
CV_DIR.mkdir(exist_ok=True)

def make_cv(task, n_splits, seed):
    if task == "classification":
        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return KFold(n_splits=n_splits, shuffle=True, random_state=seed)

def stat_value(ep, *names):
    """First non-missing value among possible column names: stats files from different runs may use
    'ENet ...' or 'Elastic Net ...'. Falls back to the endpoint's own stats_<endpoint>.csv."""
    for n in names:
        if n in stats.columns and pd.notna(stats.loc[ep, n]):
            return stats.loc[ep, n]
    f = MODELS_DIR / ep / f"stats_{ep}.csv"
    if f.exists():
        s_ep = pd.read_csv(f)
        for n in names:
            if n in s_ep.columns and pd.notna(s_ep[n].iloc[0]):
                return s_ep[n].iloc[0]
    return np.nan

def cv_predictions(ep):
    d = endpoint_data(ep)
    cache = CV_DIR / f"cv_predictions_{ep}.csv"
    if cache.exists() and not RECOMPUTE_CV:
        c = pd.read_csv(cache, index_col=0)
        c.index = c.index.astype(str)
        if list(c.index) == list(d["y_raw"].index):
            logging.info(f" - {ep}: cached CV predictions loaded")
            return d, c["predicted"].values
    X, y = d["X"].values, d["y_model"]
    l1 = pd.to_numeric(stat_value(ep, "Elastic Net best l1_ratio", "ENet best l1_ratio"), errors="coerce")
    if pd.isna(l1):
        l1 = [0.1, 0.3, 0.5, 0.7, 0.9] # not found: tuned by CV together with alpha
        logging.warning(f" - {ep}: best l1_ratio not found in the statistics files; tuned by CV instead")
    else:
        l1 = float(l1)
    alpha_max = np.max(np.abs(X.T @ (y - y.mean()))) / (len(y) * 0.1) # as in part 2 (smallest l1_ratio in grid = 0.1)
    alphas = np.logspace(np.log10(alpha_max) - 3, np.log10(alpha_max), 60)[::3]
    model = ElasticNetCV(l1_ratio=l1, alphas=alphas, cv=make_cv(d["task"], 3, GLOBAL_SEED), max_iter=10000)
    t0 = datetime.now()
    pred = cross_val_predict(model, X, y, cv=make_cv(d["task"], N_SPLITS, GLOBAL_SEED), n_jobs=-1)
    pd.DataFrame({"measured_model_scale": y, "predicted": pred}, index=d["y_raw"].index).to_csv(cache)
    logging.info(f" - {ep}: CV predictions computed in {(datetime.now() - t0).seconds} s")
    return d, pred

def cv_mean_txt(ep):
    v = pd.to_numeric(stat_value(ep, "Elastic Net CV metric mean", "ENet CV metric mean"), errors="coerce")
    return "n/a" if pd.isna(v) else f"{v:.2f}"

eps2 = [e for e in INCLUDED if e in fsaci]
if not eps2:
    raise RuntimeError("No endpoint is used in the compound ranking: nothing to plot")
ncols = 2 if len(eps2) == 4 else min(3, len(eps2))
nrows = int(np.ceil(len(eps2) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(FIG_WIDTH_FULL if ncols == 3 else FIG_WIDTH_FULL * 0.7, 2.35 * nrows), squeeze=False)
notes2 = []
for i, (ax, ep) in enumerate(zip(axes.flat, eps2)):
    d, pred = cv_predictions(ep)
    y, col = d["y_model"], endpoint_color(ep)
    if d["task"] == "classification":
        fpr, tpr, _ = roc_curve(y, pred)
        auc = roc_auc_score(y, pred)
        ax.plot([0, 1], [0, 1], color=INK2, lw=0.8, ls=(0, (3, 2)))
        ax.plot(fpr, tpr, color=col, lw=2, drawstyle="steps-post", solid_capstyle="round")
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        n_act = int(y.sum())
        ax.text(0.97, 0.05, f"AUC = {auc:.2f}\n{n_act} active / {len(y) - n_act} inactive", transform=ax.transAxes,
                ha="right", va="bottom", fontsize=6.5)
        # notes2.append(f"{ENDPOINT_LABELS.get(ep, ep)}: AUC = {auc:.2f} (this CV run; mean of part-1 repeats {cv_mean_txt(ep)})")
        notes2.append(f"{ENDPOINT_LABELS.get(ep, ep)}: AUC = {auc:.2f} (this CV run; mean of part-2 repeats {cv_mean_txt(ep)})")
    else:
        q2 = r2_score(y, pred)
        lo, hi = min(y.min(), pred.min()), max(y.max(), pred.max())
        pad = 0.05 * (hi - lo)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=INK2, lw=0.8, ls=(0, (3, 2)), zorder=1)
        ax.scatter(y, pred, s=12, color=col, alpha=0.8, edgecolors="white", linewidths=0.4, zorder=2)
        ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("Measured: " + d["y_label"]); ax.set_ylabel("Predicted (cross-validated)")
        ax.text(0.03, 0.97, f"Q² = {q2:.2f}\nn = {len(y)}", transform=ax.transAxes, ha="left", va="top", fontsize=6.5)
        # notes2.append(f"{ENDPOINT_LABELS.get(ep, ep)}: Q² = {q2:.2f} (this CV run; mean of part-1 repeats {cv_mean_txt(ep)})")
        notes2.append(f"{ENDPOINT_LABELS.get(ep, ep)}: Q² = {q2:.2f} (this CV run; mean of part-2 repeats {cv_mean_txt(ep)})")
    ax.set_title(ENDPOINT_LABELS.get(ep, ep), loc="left", fontweight="bold")
    ax.xaxis.label.set_size(6.5)
    ax.text(-0.18, 1.06, "abcdefghij"[i], transform=ax.transAxes, fontweight="bold", fontsize=9, va="bottom")
for ax in list(axes.flat)[len(eps2):]:
    ax.set_visible(False)
fig.tight_layout(w_pad=1.5, h_pad=1.5)
# save_fig(fig, "fig2_measured_vs_predicted")
save_fig(fig, "fig2_measured_vs_predicted", shared=True) # same with or without unknown features
plt.show()

save_notes("Fig 2", f"""
Cross-validated Elastic Net predictions for the endpoints used in the compound ranking ({N_SPLITS}-fold CV, one repeat;
alpha re-tuned within each training fold). Regression endpoints: measured vs. predicted on the modeling scale, dashed
line = 1:1. Classification endpoints: ROC curves, dashed line = chance.
{chr(10).join(notes2)}
""")
logging.info(f"Done Fig 2 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Fig 3: Chemical class enrichment among the prioritized compounds
# Per endpoint: top N_TOP_ENRICH compounds by FSACI (FSACI > 0) vs. all features used for that endpoint.
# Enrichment = share of a class among the top compounds / share among all features (colored only if > 1);
# one-sided hypergeometric (Fisher) test for over-representation, Benjamini-Hochberg across all cells.
# IDENTIFIED_ONLY = True: top compounds AND background both restricted to identified compounds
majorsep()
logging.info(f"Fig 3: class enrichment | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

def bh(p):
    p = np.asarray(p, float); q = np.full(p.shape, np.nan); ok = ~np.isnan(p); m = ok.sum()
    if m:
        o = np.argsort(p[ok]); r = p[ok][o] * m / np.arange(1, m + 1)
        r = np.minimum.accumulate(r[::-1])[::-1]; tmp = np.empty(m); tmp[o] = np.minimum(r, 1); q[ok] = tmp
    return q

eps3 = [e for e in INCLUDED if e in fsaci]
cls_of = lambda ids: meta["class"].reindex(ids).fillna("Unclassified").values
fsaci3 = {e: fsaci[e][keep_mask(fsaci[e]["ML_feature_id"])] for e in eps3} # all or identified only
# bg_counts = pd.DataFrame({e: pd.Series(cls_of(fsaci[e]["ML_feature_id"])).value_counts() for e in eps3}).fillna(0)
bg_counts = pd.DataFrame({e: pd.Series(cls_of(fsaci3[e]["ML_feature_id"])).value_counts() for e in eps3}).fillna(0)
big = bg_counts.index[(bg_counts.mean(axis=1) >= MIN_CLASS_SIZE) & (bg_counts.index != "Unclassified")].tolist()
def cls_group(c):
    return c if c in big or c == "Unclassified" else "Other classes"

rows = []
for e in eps3:
    # f = fsaci[e].copy()
    f = fsaci3[e].copy()
    f["cls"] = [cls_group(c) for c in cls_of(f["ML_feature_id"])]
    top = f[f["FSACI"] > 0].sort_values("FSACI", ascending=False).head(N_TOP_ENRICH)
    N, n = len(f), len(top)
    for c, K in f["cls"].value_counts().items():
        k = int((top["cls"] == c).sum())
        fe = (k / n) / (K / N) if n and K else np.nan
        rows.append(dict(endpoint=e, cls=c, k=k, n_top=n, K=int(K), N=N, fold_enrichment=fe,
                         p=hypergeom.sf(k - 1, N, K, n) if n else np.nan))
enr = pd.DataFrame(rows)
enr["q"] = bh(enr["p"])
enr["log2_fe"] = np.log2(enr["fold_enrichment"].where(enr["fold_enrichment"] > 0)).clip(0, 3) # only over-representation colored
enr.loc[enr["k"] == 0, "log2_fe"] = 0
# enr.to_csv(FIG_DIR / "fig3_class_enrichment_table.csv", index=False)
enr.to_csv(FIG_DIR / f"fig3_class_enrichment_table{FIG_SUFFIX}.csv", index=False)

n_top_all = enr.groupby("endpoint")["n_top"].first()
dropped3 = [e for e in eps3 if n_top_all.get(e, 0) == 0]
if dropped3:
    logging.warning(f" - No prioritized {'identified ' if IDENTIFIED_ONLY else ''}compounds for {dropped3}: column left out")
eps3 = [e for e in eps3 if e not in dropped3]
if not eps3:
    raise ValueError("No endpoint has prioritized compounds to test for class enrichment")
L = enr.pivot(index="cls", columns="endpoint", values="log2_fe").reindex(columns=eps3)
Kt = enr.pivot(index="cls", columns="endpoint", values="k").reindex(columns=eps3)
Qt = enr.pivot(index="cls", columns="endpoint", values="q").reindex(columns=eps3)
order_rows = [c for c in L.max(axis=1).sort_values(ascending=False).index if c not in ("Other classes", "Unclassified")]
order_rows += [c for c in ["Other classes", "Unclassified"] if c in L.index]
L, Kt, Qt = L.loc[order_rows], Kt.loc[order_rows], Qt.loc[order_rows]
n_top_by_ep = enr.groupby("endpoint")["n_top"].first()

fig_w = min(FIG_WIDTH_FULL, 2.4 + 0.62 * len(eps3))
fig, ax = plt.subplots(figsize=(fig_w, 0.24 * len(L) + 1.3))
im = ax.imshow(L.values.astype(float), cmap=ENRICH_CMAP, vmin=0, vmax=3, aspect="auto")
for i in range(L.shape[0]):
    for j in range(L.shape[1]):
        v, k, q = L.iat[i, j], Kt.iat[i, j], Qt.iat[i, j]
        if pd.isna(k):
            continue
        ax.text(j, i, f"{int(k)}{'*' if q < 0.05 else ''}", ha="center", va="center", fontsize=6,
                color="white" if (not pd.isna(v) and v >= 1.8) else INK)
ax.set_xticks(range(len(eps3)))
ax.set_xticklabels([f"{ENDPOINT_LABELS.get(e, e)}\n(top {int(n_top_by_ep[e])})" for e in eps3])
if IDENTIFIED_ONLY:
    ax.set_title("Identified compounds only", loc="left", fontsize=6.5, color=INK2, pad=28)
ax.xaxis.tick_top(); ax.xaxis.set_label_position("top")
mean_k = bg_counts.reindex(index=big).mean(axis=1)
ax.set_yticks(range(len(L)))
ax.set_yticklabels(L.index)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xticks(np.arange(-0.5, len(eps3)), minor=True); ax.set_yticks(np.arange(-0.5, len(L)), minor=True)
ax.grid(which="minor", color="white", linewidth=1.5); ax.tick_params(which="minor", length=0)
cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.04, pad=0.03, aspect=30, shrink=0.8, ticks=[0, 1, 2, 3])
cb.ax.set_xticklabels(["≤ 1", "2", "4", "≥ 8"])
# cb.set_label("Fold enrichment (share among top compounds / share among all features)", fontsize=6.5)
cb.set_label(f"Fold enrichment (share among top compounds / share among all {'identified ' if IDENTIFIED_ONLY else ''}features)", fontsize=6.5)
cb.outline.set_visible(False); cb.ax.tick_params(labelsize=6, length=2)
save_fig(fig, "fig3_class_enrichment")
plt.show()

sig = enr[(enr["q"] < 0.05) & (enr["fold_enrichment"] > 1)].sort_values("q")
sig_txt = "; ".join(f"{r.cls} for {ENDPOINT_LABELS.get(r.endpoint, r.endpoint)} ({r.k}/{r.n_top}, {r.fold_enrichment:.1f}-fold, q = {r.q:.2g})"
                    for r in sig.itertuples()) or "none"
logging.info(f" - Significant enrichments (q < 0.05): {sig_txt}")
save_notes("Fig 3", f"""
Chemical class enrichment among the top {N_TOP_ENRICH} prioritized compounds (FSACI > 0) per endpoint, relative to all
features used for that endpoint. Color: fold enrichment (gray = not over-represented; capped at 8-fold);
numbers: compounds of that class among the top compounds; * = q < 0.05 (one-sided hypergeometric test,
Benjamini-Hochberg). Classes with < {MIN_CLASS_SIZE} features pooled as "Other classes".
{"Identified compounds only: unknown features removed from both the top compounds and the background." if IDENTIFIED_ONLY else "All features, including unknown features."}
Significant: {sig_txt}.
""")
logging.info(f"Done Fig 3 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Fig 4: Endpoint vs. concentration for the top compounds of each endpoint (one figure per endpoint)
# Regression: endpoint (modeling scale) vs. log10 concentration; n.d. = not detected, open = below-LOD estimate.
# Classification: log10 concentration in inactive vs. active samples.
# NOTE for the caption: compounds were selected on the same data, so single-compound associations look stronger
# than in independent data.
# IDENTIFIED_ONLY = True: top N_TOP_COMPOUNDS identified compounds (unknowns skipped); title shows the overall FSACI rank
majorsep()
logging.info(f"Fig 4: endpoint vs. concentration | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

rng = np.random.default_rng(GLOBAL_SEED)

def conc_axis_values(fid, idx):
    conc = DUST_CONC.loc[idx, fid]
    det, below = DET.loc[idx, fid].values, BELOW_LOD.loc[idx, fid].values
    if NEG_HANDLING != "abs":
        below = np.zeros_like(below)
    pos = conc.values > 0
    logc = np.full(len(conc), np.nan)
    logc[pos] = np.log10(conc.values[pos])
    return conc.values, logc, det, below, ~pos

def nd_position(logc, span_frac=0.14):
    v = logc[~np.isnan(logc)]
    if len(v) == 0:
        return 0.0, 1.0
    lo, hi = v.min(), v.max()
    span = hi - lo if hi > lo else 1.0
    return lo - span_frac * span - 0.05 * span, span

notes4 = []
for ep in [e for e in INCLUDED if e in fsaci]:
    d = endpoint_data(ep)
    idx, y, col = d["y_raw"].index, d["y_model"], endpoint_color(ep)
    # top = fsaci[ep][fsaci[ep]["FSACI"] > 0].sort_values("FSACI", ascending=False).head(N_TOP_COMPOUNDS)
    ranked = fsaci[ep][fsaci[ep]["FSACI"] > 0].sort_values("FSACI", ascending=False).copy()
    ranked["overall_rank"] = np.arange(1, len(ranked) + 1) # FSACI rank among all prioritized features
    top = ranked[keep_mask(ranked["ML_feature_id"])].head(N_TOP_COMPOUNDS)
    if top.empty:
        # logging.warning(f" - {ep}: no prioritized compounds")
        logging.warning(f" - {ep}: no prioritized {'identified ' if IDENTIFIED_ONLY else ''}compounds; figure skipped")
        continue
    ncols = 3
    nrows = int(np.ceil(len(top) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(FIG_WIDTH_FULL, 2.35 * nrows), squeeze=False)
    for k, (ax, (_, row)) in enumerate(zip(axes.flat, top.iterrows()), start=1):
        fid = row["ML_feature_id"]
        conc, logc, det, below, nd = conc_axis_values(fid, idx)
        cls = meta["class"].get(fid, "Unclassified")
        ctype = meta["concentration_type"].get(fid, "") if "concentration_type" in meta.columns else ""
        if d["task"] == "classification":
            x_nd, span = nd_position(logc)
            act = y == 1
            for gx, mask in [(0, ~act), (1, act)]:
                jit = rng.uniform(-0.17, 0.17, mask.sum())
                lv = np.where(np.isnan(logc[mask]), x_nd, logc[mask])
                m_det, m_below, m_nd = det[mask], below[mask], nd[mask]
                ax.scatter(gx + jit[m_det], lv[m_det], s=10, color=col, alpha=0.8, edgecolors="white", linewidths=0.3, zorder=3)
                ax.scatter(gx + jit[m_below], lv[m_below], s=10, facecolors="white", edgecolors=col, linewidths=0.7, zorder=2)
                ax.scatter(gx + jit[m_nd], lv[m_nd], s=10, facecolors="white", edgecolors=EXCLUDED, linewidths=0.7, zorder=2)
                med = np.nanmedian(logc[mask][det[mask]]) if det[mask].any() else np.nan
                if not np.isnan(med):
                    ax.plot([gx - 0.28, gx + 0.28], [med, med], color=INK, lw=1.2, zorder=4)
            ax.axhspan(x_nd - 0.06 * span, x_nd + 0.06 * span, color=NEUTRAL, zorder=0)
            ticks = MaxNLocator(4).tick_values(np.nanmin(logc), np.nanmax(logc)) if np.isfinite(np.nanmax(logc)) else []
            ticks = [t for t in ticks if np.nanmin(logc) - 0.05 * span <= t <= np.nanmax(logc) + 0.05 * span]
            ax.set_yticks([x_nd] + list(ticks)); ax.set_yticklabels(["n.d."] + [f"{t:g}" for t in ticks])
            ax.set_xticks([0, 1]); ax.set_xticklabels([f"inactive\n(n = {(~act).sum()})", f"active\n(n = {act.sum()})"])
            ax.set_xlim(-0.55, 1.55)
            auc = mannwhitneyu(conc[act], conc[~act]).statistic / (act.sum() * (~act).sum())
            stat_txt = f"AUC = {auc:.2f}; detected {det[~act].mean():.0%} vs. {det[act].mean():.0%}"
            ax.set_ylabel(f"log10 conc. ({ctype})" if ctype else "log10 concentration", fontsize=6)
            notes4.append(f"{ENDPOINT_LABELS.get(ep, ep)} #{k} {feat_name(fid)}: AUC = {auc:.2f}, detected {det[~act].mean():.0%} (inactive) vs. {det[act].mean():.0%} (active)")
        else:
            x_nd, span = nd_position(logc)
            xv = np.where(np.isnan(logc), x_nd + rng.uniform(-0.03, 0.03, len(logc)) * span, logc)
            ax.axvspan(x_nd - 0.06 * span, x_nd + 0.06 * span, color=NEUTRAL, zorder=0)
            ax.scatter(xv[det], y[det], s=10, color=col, alpha=0.8, edgecolors="white", linewidths=0.3, zorder=3)
            ax.scatter(xv[below], y[below], s=10, facecolors="white", edgecolors=col, linewidths=0.7, zorder=2)
            ax.scatter(xv[nd], y[nd], s=10, facecolors="white", edgecolors=EXCLUDED, linewidths=0.7, zorder=2)
            okfit = det & ~np.isnan(logc)
            if okfit.sum() >= 10 and np.ptp(logc[okfit]) > 0:
                b1, b0 = np.polyfit(logc[okfit], y[okfit], 1)
                xs = np.array([logc[okfit].min(), logc[okfit].max()])
                ax.plot(xs, b0 + b1 * xs, color=INK, lw=1.0, zorder=4)
            rho, p = spearmanr(conc, y)
            stat_txt = f"ρ = {rho:.2f}, p = {p:.1g}; detected {det.sum()}/{len(det)}"
            ticks = MaxNLocator(4).tick_values(np.nanmin(logc), np.nanmax(logc)) if (~np.isnan(logc)).any() else []
            ticks = [t for t in ticks if np.nanmin(logc) - 0.05 * span <= t <= np.nanmax(logc) + 0.05 * span]
            ax.set_xticks([x_nd] + list(ticks)); ax.set_xticklabels(["n.d."] + [f"{t:g}" for t in ticks])
            ax.set_xlabel(f"log10 conc. ({ctype})" if ctype else "log10 concentration", fontsize=6)
            notes4.append(f"{ENDPOINT_LABELS.get(ep, ep)} #{k} {feat_name(fid)}: Spearman rho = {rho:.2f} (p = {p:.1g}), detected {det.sum()}/{len(det)}")
        name = "\n".join(textwrap.wrap(feat_name(fid, 70), 36)[:2])
        ax.set_title(f"{k}. {name}", loc="left", fontsize=6.5, fontweight="bold", pad=17)
        rank_txt = f" · FSACI rank {int(row['overall_rank'])}" if IDENTIFIED_ONLY else ""
        # ax.text(0, 1.012, f"{textwrap.shorten(str(cls), 45, placeholder='…')}\n{stat_txt}", transform=ax.transAxes,
        ax.text(0, 1.012, f"{textwrap.shorten(str(cls), 40, placeholder='…')}{rank_txt}\n{stat_txt}", transform=ax.transAxes,
                fontsize=6, color=INK2, va="bottom", ha="left", linespacing=1.25)
    for ax in list(axes.flat)[len(top):]:
        ax.set_visible(False)
    if d["task"] != "classification":
        fig.supylabel(d["y_label"], fontsize=7, x=0.005)
    handles = [Line2D([], [], marker="o", ls="", ms=3.5, mfc=col, mec="white", label="detected"),
               Line2D([], [], marker="o", ls="", ms=3.5, mfc="white", mec=EXCLUDED, label="not detected (n.d.)")]
    if NEG_HANDLING == "abs" and BELOW_LOD[top["ML_feature_id"].tolist()].values.any():
        handles.insert(1, Line2D([], [], marker="o", ls="", ms=3.5, mfc="white", mec=col, label="below-LOD estimate"))
    if d["task"] == "classification":
        handles.append(Line2D([], [], color=INK, lw=1.2, label="median of detected"))
    else:
        handles.append(Line2D([], [], color=INK, lw=1.0, label="linear fit (detected)"))
    fig.suptitle(f"{ENDPOINT_LABELS.get(ep, ep)}: top {len(top)} prioritized {'identified ' if IDENTIFIED_ONLY else ''}compounds", x=0.01, ha="left",
                 fontsize=7.5, fontweight="bold")
    fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.0, 1.0), ncol=len(handles), frameon=False)
    fig.tight_layout(rect=(0.01, 0, 1, 0.96), h_pad=1.2, w_pad=1.0)
    # fig_h = fig.get_size_inches()[1]
    # fig.suptitle(f"{ENDPOINT_LABELS.get(ep, ep)}: top {len(top)} prioritized {'identified ' if IDENTIFIED_ONLY else ''}compounds",
    #              x=0.01, y=1 - 0.04 / fig_h, ha="left", va="top", fontsize=8, fontweight="bold")
    # fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.005, 1 - 0.20 / fig_h), ncol=len(handles),
    #            frameon=False, handletextpad=0.3, columnspacing=1.2)
    # fig.tight_layout(rect=(0.01, 0, 1, 1 - 0.42 / fig_h), h_pad=1.2, w_pad=1.0)
    
    save_fig(fig, f"fig4_{ep}_top_compounds")
    plt.show()

save_notes("Fig 4", """
Endpoint vs. concentration of the top prioritized compounds (highest FSACI) per endpoint. Regression endpoints:
endpoint on the modeling scale vs. log10 concentration; filled = detected, open colored = below-LOD estimate,
open gray at "n.d." = not detected; line = linear fit to detected values; Spearman rho over all samples
(non-detects ranked lowest). Single-compound associations can be weak or even negative for a compound that the
multivariate model ranks high, because the model credits each compound given all others. Classification endpoints: log10 concentration in inactive vs. active samples;
bar = median of detected values; AUC = probability that an active sample has a higher concentration than an
inactive one. Compounds were selected on the same data, so the single-compound associations shown are
optimistic and illustrative; they do not establish causality.
""" + ("Identified compounds only: unknown features skipped; 'FSACI rank' = rank among all prioritized features.\n" if IDENTIFIED_ONLY else "")
  + "\n".join(notes4))
logging.info(f"Done Fig 4 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Fig 5: Cross-endpoint heatmap of the top compounds (global ranking)
majorsep()
logging.info(f"Fig 5: cross-endpoint heatmap | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

if global_summary is None or global_summary.empty:
    raise FileNotFoundError("compound_global_summary_w_metadata.csv missing or empty: run part 2 first")

# gs = global_summary.sort_values("global_rank").head(N_TOP_HEATMAP).copy()
# gs["ML_feature_id"] = gs["ML_feature_id"].astype(str).str.strip()
gs = global_summary.sort_values("global_rank").copy()
gs["ML_feature_id"] = gs["ML_feature_id"].astype(str).str.strip()
n_unknown_skipped = int((~keep_mask(gs["ML_feature_id"].head(N_TOP_HEATMAP))).sum())
gs = gs[keep_mask(gs["ML_feature_id"])].head(N_TOP_HEATMAP) # all, or identified only (global rank numbers kept)
if gs.empty:
    raise ValueError("No compounds left to plot (IDENTIFIED_ONLY = True and no identified compound in the global ranking)")
cols5 = [e for e in endpoints_all if f"score_{e}" in gs.columns]
S = gs[[f"score_{e}" for e in cols5]].astype(float).values
R = gs[[f"rank_{e}" for e in cols5]].apply(pd.to_numeric, errors="coerce").values
names = [feat_name(f, 48) for f in gs["ML_feature_id"]]
classes = [textwrap.shorten(str(meta["class"].get(f, "Unclassified")), 30, placeholder="…") for f in gs["ML_feature_id"]]
n = len(gs)

fig = plt.figure(figsize=(FIG_WIDTH_FULL, 0.19 * n + 1.35))
gsp = fig.add_gridspec(1, 3, width_ratios=[1.35, 0.42 * len(cols5), 1.1], wspace=0.04)
ax_c, ax_h, ax_b = fig.add_subplot(gsp[0]), fig.add_subplot(gsp[1]), fig.add_subplot(gsp[2])

# Class column + compound names
ax_c.set_ylim(n - 0.5, -0.5); ax_c.set_xlim(0, 1)
for i, c in enumerate(classes):
    ax_c.text(0.98, i, c, ha="right", va="center", fontsize=6, color=INK2)
ax_c.set_yticks(range(n)); ax_c.set_yticklabels([f"{r}. {nm}" for r, nm in zip(gs["global_rank"], names)], fontsize=6)
ax_c.set_xticks([]); ax_c.tick_params(axis="y", length=0)
for s in ax_c.spines.values():
    s.set_visible(False)
ax_c.set_title("Class", loc="right", fontsize=6.5, color=INK2)
if IDENTIFIED_ONLY:
    fig.suptitle("Identified compounds only (numbers = global rank among all features)", x=0.01, y=1.0, ha="left",
                 va="bottom", fontsize=6.5, color=INK2)

# Heatmap: endpoint score (1 = rank 1), number = FSACI rank, gray = not in the endpoint's top set
cmap = SEQ_CMAP.copy(); cmap.set_bad(NEUTRAL)
im = ax_h.imshow(np.ma.masked_where(S <= 0, S), cmap=cmap, vmin=0, vmax=1, aspect="auto")
for i in range(n):
    for j in range(len(cols5)):
        if S[i, j] > 0 and not np.isnan(R[i, j]):
            ax_h.text(j, i, f"{int(R[i, j])}", ha="center", va="center", fontsize=5.5,
                      color="white" if S[i, j] >= 0.55 else INK)
ax_h.set_xticks(range(len(cols5))); ax_h.set_xticklabels([ENDPOINT_LABELS.get(e, e) for e in cols5], rotation=45, ha="left")
ax_h.xaxis.tick_top(); ax_h.set_yticks([])
ax_h.set_xticks(np.arange(-0.5, len(cols5)), minor=True); ax_h.set_yticks(np.arange(-0.5, n), minor=True)
ax_h.grid(which="minor", color="white", linewidth=1.2); ax_h.tick_params(which="both", length=0)
for s in ax_h.spines.values():
    s.set_visible(False)

# Global priority score
ax_b.barh(range(n), gs["global_priority_score"].values, height=0.62, color=SERIES[0], zorder=2)
ax_b.set_ylim(n - 0.5, -0.5); ax_b.set_yticks([])
ax_b.set_xlim(0, max(gs["global_priority_score"].max() * 1.08, 0.05))
ax_b.xaxis.set_major_locator(MaxNLocator(3)); ax_b.grid(axis="x", zorder=0)
ax_b.spines["left"].set_visible(False)
ax_b.set_xlabel("Global priority score", fontsize=6.5)
ax_b.xaxis.set_label_position("top"); ax_b.xaxis.tick_top()
ax_b.spines["bottom"].set_visible(False); ax_b.spines["top"].set_visible(True)

pos_h = ax_h.get_position()
cax = fig.add_axes([pos_h.x0, pos_h.y0 - 0.45 / fig.get_figheight(), pos_h.width, 0.10 / fig.get_figheight()])
cb = fig.colorbar(im, cax=cax, orientation="horizontal")
cb.set_label("Endpoint score (1 = rank 1; number = FSACI rank; gray = not in top set)", fontsize=6)
cb.outline.set_visible(False); cb.ax.tick_params(labelsize=5.5, length=2)
save_fig(fig, "fig5_cross_endpoint_heatmap")
plt.show()

multi = int((gs["n_blocks_hit"] >= 2).sum()) if "n_blocks_hit" in gs.columns else np.nan
save_notes("Fig 5", f"""
Top {n} compounds of the cross-endpoint ranking. Cells: endpoint score (1 = rank 1 within that endpoint's top
compounds), numbers = FSACI rank for the endpoint, gray = not among the endpoint's top compounds. Right: global
priority score (weighted mean of block scores; weights = model quality). {multi} of the {n} compounds rank high
in two or more endpoint blocks.
{f"Identified compounds only: unknown features removed; numbers before the names are the global ranks among all features ({n_unknown_skipped} unknown features were among the overall top {N_TOP_HEATMAP})." if IDENTIFIED_ONLY else "All features, including unknown features."}
""")
logging.info(f" - Compounds hitting >= 2 blocks: {multi} of {n}")
if IDENTIFIED_ONLY:
    logging.info(f" - Unknown features among the overall top {N_TOP_HEATMAP}: {n_unknown_skipped} (removed)")
# logging.info(f" - Caption drafts: {FIG_DIR / 'figure_notes.txt'}")
logging.info(f" - Caption drafts: {FIG_DIR / f'figure_notes{FIG_SUFFIX}.txt'}")
logging.info(f"Done Fig 5 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()
